# Imports and config


In [ ]:
import gc
import os
import re
import urllib.request
import zipfile
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import ParameterGrid, train_test_split
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer


# Data Loading


In [ ]:
df = pd.read_csv("imdb.csv")
print(df.head())

In [ ]:
# Remove HTML tags from reviews
df["review"] = df["review"].str.replace(r"<[^>]+>", "", regex=True)

# Map sentiment strings to integers
df["sentiment"] = df["sentiment"].map({"positive": 1, "negative": 0})

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["sentiment"]
)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df["sentiment"]
)

# Data Exploration


In [ ]:
print(train_df["sentiment"].value_counts())

# RNNs


## Tokenizer and vocabulary


In [ ]:
class TextDataFrameDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):  # ty:ignore[invalid-method-override]
        row = self.dataframe.iloc[idx]
        return row["sentiment"], row["review"]


def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'([.?!,\'"])', r" \1 ", text)
    return text.split()


class Vocab:
    def __init__(self, text_iterable, min_freq=1):
        self.stoi = {"<unk>": 0, "<pad>": 1}
        self.unk_idx = 0
        self.pad_idx = 1

        counter = Counter()
        for text in text_iterable:
            counter.update(tokenize(text))

        idx = 2
        for word, freq in counter.items():
            if freq >= min_freq:
                self.stoi[word] = idx
                idx += 1

    def __call__(self, tokens):
        return [self.stoi.get(token, self.unk_idx) for token in tokens]

    def __len__(self):
        return len(self.stoi)


def create_collate_fn(vocab, max_len=512):
    def collate_batch(batch):
        label_list, text_list, lengths_list = [], [], []

        for label, text in batch:
            label_list.append(label)
            tokens = vocab(tokenize(text))
            tokens = tokens[:max_len]
            processed_text = torch.tensor(tokens, dtype=torch.int64)
            text_list.append(processed_text)
            lengths_list.append(len(tokens))

        labels = torch.tensor(label_list, dtype=torch.float32)
        lengths = torch.tensor(lengths_list, dtype=torch.int64)

        padded_texts = pad_sequence(
            text_list, batch_first=True, padding_value=vocab.pad_idx
        )

        return labels, padded_texts, lengths

    return collate_batch

In [ ]:
vocab = Vocab(train_df["review"], min_freq=1)

train_dataset = TextDataFrameDataset(train_df)
val_dataset = TextDataFrameDataset(val_df)
test_dataset = TextDataFrameDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=create_collate_fn(vocab),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=create_collate_fn(vocab),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=create_collate_fn(vocab),
)

## LSTM and GRU


In [ ]:
class ClassicLSTM(nn.Module):
    def __init__(self, n_layers, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=pad_idx
        )
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, text, lengths):
        embedded = self.embedding(text)

        packed_embedded = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_output, (h_n, c_n) = self.lstm(packed_embedded)

        final_hidden_state = h_n[-1]

        return self.fc(final_hidden_state).squeeze()

In [ ]:
class ClassicGRU(nn.Module):
    def __init__(self, n_layers, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=pad_idx
        )
        self.gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, text, lengths):
        embedded = self.embedding(text)

        packed_embedded = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_output, h_n = self.gru(packed_embedded)

        final_hidden_state = h_n[-1]
        return self.fc(final_hidden_state).squeeze()

In [ ]:
def train(model, title, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    criterion = nn.BCEWithLogitsLoss()
    criterion = criterion.to(device)

    def binary_accuracy(preds, y):
        rounded_preds = torch.round(torch.sigmoid(preds))
        correct = (rounded_preds == y).float()
        return correct.sum() / len(correct)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        epoch_train_loss = 0
        epoch_train_acc = 0

        model.train()

        for batch_labels, batch_texts, batch_lengths in train_loader:
            batch_labels = batch_labels.to(device)
            batch_texts = batch_texts.to(device)

            optimizer.zero_grad()
            predictions = model(batch_texts, batch_lengths)

            loss = criterion(predictions, batch_labels)
            acc = binary_accuracy(predictions, batch_labels)

            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()
            epoch_train_acc += acc.item()

        epoch_val_loss = 0
        epoch_val_acc = 0

        model.eval()

        with torch.no_grad():
            for batch_labels, batch_texts, batch_lengths in val_loader:
                batch_labels = batch_labels.to(device)
                batch_texts = batch_texts.to(device)

                predictions = model(batch_texts, batch_lengths)

                loss = criterion(predictions, batch_labels)
                acc = binary_accuracy(predictions, batch_labels)

                epoch_val_loss += loss.item()
                epoch_val_acc += acc.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_train_acc = epoch_train_acc / len(train_loader)

        avg_val_loss = epoch_val_loss / len(val_loader)
        avg_val_acc = epoch_val_acc / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        print(
            f"Epoch: {epoch + 1:02} | "
            f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc * 100:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc * 100:.2f}%"
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    fig.suptitle(title)
    axes[0].plot(history["train_loss"], label="Training")
    axes[0].plot(history["val_loss"], label="Validation", linestyle="--")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Error")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Training")
    axes[1].plot(history["val_acc"], label="Validation", linestyle="--")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    return model, history

In [ ]:
INPUT_DIM = len(vocab)
EMBED_DIM = 100
HIDDEN_DIM = 256
N_LAYERS = 2
PAD_IDX = vocab.pad_idx
model = ClassicLSTM(N_LAYERS, INPUT_DIM, EMBED_DIM, HIDDEN_DIM, PAD_IDX)
train(model, "LSTM")

INPUT_DIM = len(vocab)
EMBED_DIM = 100
HIDDEN_DIM = 256
N_LAYERS = 2
PAD_IDX = vocab.pad_idx
model2 = ClassicGRU(N_LAYERS, INPUT_DIM, EMBED_DIM, HIDDEN_DIM, PAD_IDX)
train(model2, "GRU")

## LSTM with hidden state pooling


In [ ]:
class PoolingLSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        n_layers,
        bidirectional,
        dropout,
        pad_idx,
    ):
        super().__init__()
        self.pad_idx = pad_idx

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True,
        )

        fc_input_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(fc_input_dim, 1)

    def forward(self, text, lengths):
        mask = (text != self.pad_idx).float().unsqueeze(-1)

        embedded = self.embedding(text)

        output, _ = self.lstm(embedded)

        masked_output = output * mask

        summed_states = torch.sum(masked_output, dim=1)

        real_lengths = lengths.to(summed_states.device).float().view(-1, 1)
        mean_pooled = summed_states / real_lengths

        return self.fc(mean_pooled).squeeze()

### Hyperparameter tuning


In [ ]:
param_grid = {
    "bidirectional": [True, False],
    "hidden_dim": [100, 200],
    "embed_dim": [75, 150],
}

for params in ParameterGrid(param_grid):
    print(f"Training hidden state pooling LSTM with params: {params}")

    model = PoolingLSTM(
        vocab_size=len(vocab),
        dropout=0.25,
        n_layers=2,
        pad_idx=vocab.pad_idx,
        **params,
    )

    train(model, f"PoolingLSTM {params}", epochs=2)

    del model
    torch.cuda.empty_cache()

## LSTM wiht GloVe embeddings

In [ ]:
glove_zip = "glove.6B.zip"
glove_file = "glove.6B.100d.txt"

if not os.path.exists(glove_file):
    urllib.request.urlretrieve("https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip", glove_zip)
    with zipfile.ZipFile(glove_zip, "r") as zip_ref:
        zip_ref.extractall()

glove_vectors = {}
with open(glove_file, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = torch.tensor([float(x) for x in values[1:]], dtype=torch.float32)
        glove_vectors[word] = vector

glove_embedding_matrix = torch.zeros(len(vocab), 100)

for word, idx in vocab.stoi.items():
    if word in glove_vectors:
        glove_embedding_matrix[idx] = glove_vectors[word]
    else:
        if idx != vocab.pad_idx:
            glove_embedding_matrix[idx] = torch.randn(100)

class GloVeLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, n_layers, bidirectional, dropout, pad_idx, freeze_embeddings=False):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=freeze_embeddings, padding_idx=pad_idx)
        
        embed_dim = embedding_matrix.shape[1]
        
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True,
        )
        
        fc_input_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(fc_input_dim, 1)

    def forward(self, text, lengths):
        mask = (text != self.pad_idx).float().unsqueeze(-1)
        embedded = self.embedding(text)
        output, _ = self.lstm(embedded)
        masked_output = output * mask
        summed_states = torch.sum(masked_output, dim=1)
        real_lengths = lengths.to(summed_states.device).float().view(-1, 1)
        mean_pooled = summed_states / real_lengths
        return self.fc(mean_pooled).squeeze()

def train(model, train_loader, val_loader, title, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss().to(device)

    def binary_accuracy(preds, y):
        rounded_preds = torch.round(torch.sigmoid(preds))
        correct = (rounded_preds == y).float()
        return correct.sum() / len(correct)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        epoch_train_loss = 0
        epoch_train_acc = 0

        model.train()

        for batch_labels, batch_texts, batch_lengths in train_loader:
            batch_labels = batch_labels.to(device)
            batch_texts = batch_texts.to(device)

            optimizer.zero_grad()
            predictions = model(batch_texts, batch_lengths)

            loss = criterion(predictions, batch_labels)
            acc = binary_accuracy(predictions, batch_labels)

            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()
            epoch_train_acc += acc.item()

        epoch_val_loss = 0
        epoch_val_acc = 0

        model.eval()

        with torch.no_grad():
            for batch_labels, batch_texts, batch_lengths in val_loader:
                batch_labels = batch_labels.to(device)
                batch_texts = batch_texts.to(device)

                predictions = model(batch_texts, batch_lengths)

                loss = criterion(predictions, batch_labels)
                acc = binary_accuracy(predictions, batch_labels)

                epoch_val_loss += loss.item()
                epoch_val_acc += acc.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_train_acc = epoch_train_acc / len(train_loader)

        avg_val_loss = epoch_val_loss / len(val_loader)
        avg_val_acc = epoch_val_acc / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        print(
            f"Epoch: {epoch + 1:02} | "
            f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc * 100:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc * 100:.2f}%"
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    fig.suptitle(title)
    axes[0].plot(history["train_loss"], label="Training")
    axes[0].plot(history["val_loss"], label="Validation", linestyle="--")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Error")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Training")
    axes[1].plot(history["val_acc"], label="Validation", linestyle="--")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    return model, history


model_glove = GloVeLSTM(
    embedding_matrix=glove_embedding_matrix,
    hidden_dim=100,
    n_layers=1,
    bidirectional=True,
    dropout=0.25,
    pad_idx=vocab.pad_idx,
    freeze_embeddings=False
)

train(model_glove, train_loader, val_loader, "LSTM with Pre-trained GloVe", epochs=5)

del model_glove
del glove_vectors
torch.cuda.empty_cache()
gc.collect()

# BERT Fine-tuning


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def create_bert_collate_fn(tokenizer):
    def collate_batch(batch):
        label_list, text_list = [], []

        for label, text in batch:
            label_list.append(label)
            text_list.append(text)

        labels = torch.tensor(label_list, dtype=torch.float32)

        tokenized_batch = tokenizer(
            text_list,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )

        return (
            labels,
            tokenized_batch["input_ids"],
            tokenized_batch["attention_mask"],
        )

    return collate_batch


bert_train_loader = DataLoader(
    train_dataset,
    batch_size=12,
    shuffle=True,
    collate_fn=create_bert_collate_fn(tokenizer),
)

bert_val_loader = DataLoader(
    val_dataset,
    batch_size=12,
    shuffle=True,
    collate_fn=create_bert_collate_fn(tokenizer),
)

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.BCEWithLogitsLoss().to(device)


In [ ]:
def fine_tune_bert(model, train_loader, val_loader, title, epochs=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    criterion = nn.BCEWithLogitsLoss()
    criterion = criterion.to(device)

    scaler = torch.amp.GradScaler("cuda")

    def binary_accuracy(preds, y):
        rounded_preds = torch.round(torch.sigmoid(preds))
        correct = (rounded_preds == y).float()
        return correct.sum() / len(correct)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    tl_len = len(train_loader)

    for epoch in range(epochs):
        epoch_train_loss = 0
        epoch_train_acc = 0

        model.train()

        for i, (batch_labels, input_ids, attention_mask) in enumerate(train_loader):
            batch_labels = batch_labels.to(device)
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            optimizer.zero_grad()

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(
                    input_ids=input_ids, attention_mask=attention_mask
                )
                predictions = outputs.logits.squeeze(-1)

                loss = criterion(predictions, batch_labels)
                acc = binary_accuracy(predictions, batch_labels)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            epoch_train_loss += loss.item()
            epoch_train_acc += acc.item()

            print(f"Progress: {i/tl_len}")

        epoch_val_loss = 0
        epoch_val_acc = 0

        model.eval()

        with torch.no_grad():
            for batch_labels, input_ids, attention_mask in val_loader:
                batch_labels = batch_labels.to(device)
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)

                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    outputs = model(
                        input_ids=input_ids, attention_mask=attention_mask
                    )
                    predictions = outputs.logits.squeeze(-1)

                    loss = criterion(predictions, batch_labels)
                    acc = binary_accuracy(predictions, batch_labels)

                epoch_val_loss += loss.item()
                epoch_val_acc += acc.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_train_acc = epoch_train_acc / len(train_loader)

        avg_val_loss = epoch_val_loss / len(val_loader)
        avg_val_acc = epoch_val_acc / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        print(
            f"Epoch: {epoch + 1:02} | "
            f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc * 100:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc * 100:.2f}%"
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    fig.suptitle(title)
    axes[0].plot(history["train_loss"], label="Training")
    axes[0].plot(history["val_loss"], label="Validation", linestyle="--")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Error")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Training")
    axes[1].plot(history["val_acc"], label="Validation", linestyle="--")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    return model, history


fine_tune_bert(model, bert_train_loader, bert_val_loader, "BERT", epochs=3)
